# train_v61_dinov2_lss_bevattn_kaggle

Kaggle notebook для продолжения `DINO + LSS` из старого `last.pt` с добавлением `BEV self-attention` в decoder bottleneck.


In [ ]:
%load_ext autoreload
%autoreload 2

import os, copy, time, json, math, random, gc
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from torchvision import transforms
from PIL import Image, ImageFile, ImageFilter
from tqdm import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

# Kaggle-safe dataset layout from letiti6e/bev-yandex
DATA_TRAIN = Path('/kaggle/input/bev-yandex/autonomy_yandex_dataset_train_kaggle_safe')
DATA_VAL   = Path('/kaggle/input/bev-yandex/autonomy_yandex_dataset_val_kaggle_safe')
DATA_TEST  = Path('/kaggle/input/bev-yandex/autonomy_yandex_dataset_test_kaggle_safe')

# Upload / mount your v6 last.pt in a Kaggle dataset and point here
ARTIFACTS_DIR = Path('/kaggle/input/bev-artifacts')
RESUME_CKPT = ARTIFACTS_DIR / 'last.pt'

cfg = {
    'run_dir': '/kaggle/working/runs/v62_dinov2_lss_bev2x_cleaned',
    'img_hw': (392, 700),
    'batch_size': 1,
    'val_batch_size': 1,
    'grad_accum': 24,
    'epochs': 40,
    'warmup_epochs': 0,
    'freeze_backbone_epochs': 0,
    'unfreeze_last_blocks': 2,
    'lr_backbone': 6e-6,
    'lr_image_neck': 6e-5,
    'lr_lss_bev': 8e-5,
    'weight_decay': 1e-4,
    'embed_weight_decay': 1e-2,
    'pos_weight': 5.0,
    'num_workers': 2,
    'val_num_workers': 0,
    'seed': 42,
    'ema_decay': 0.995,
    'val_target_size': 200,
    'min_rover_count': 30,
    'topk_rovers': 25,
    'rover_emb_dim': 8,
    'rover_cond_dim': 8,
    'mae_dedup_thr': 0.02,
    'dedup_camera': '/camera/inner/frontal/middle',
    'use_clean_cache': True,
    'backbone_name': 'dinov2_vitb14',
    'hub_repo': 'facebookresearch/dinov2',
    'backbone_out_dim': 768,
    'patch_size': 14,
    'tap_layers': [2, 5, 8, 11],
    'neck_dim': 128,
    'context_dim': 80,
    'depth_bins': 24,
    'depth_min': 1.0,
    'depth_max': 80.0,
    'world_z_min': -2.0,
    'world_z_max': 4.5,
    'bev_base_channels': 96,
    'bev_gn_groups': 8,
    'bev_upsample_factor': 2,
    'resume_training': True,
    'resume_ckpt': str(RESUME_CKPT),
    'use_ddp': False,
    'use_amp': True,
}

RUN_DIR = Path(cfg['run_dir'])
RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = RUN_DIR / 'preproc_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
TORCH_HUB_DIR = RUN_DIR / 'torch_hub'
TORCH_HUB_DIR.mkdir(parents=True, exist_ok=True)
torch.hub.set_dir(str(TORCH_HUB_DIR))


def get_world_size():
    return int(os.environ.get('WORLD_SIZE', '1'))


def get_rank():
    return int(os.environ.get('RANK', '0'))


def get_local_rank():
    return int(os.environ.get('LOCAL_RANK', '0'))


def is_dist_enabled():
    return cfg['use_ddp'] and get_world_size() > 1


def is_main_process():
    return get_rank() == 0


def setup_distributed():
    if not is_dist_enabled():
        return
    if dist.is_available() and not dist.is_initialized():
        dist.init_process_group(backend='nccl', init_method='env://')
        torch.cuda.set_device(get_local_rank())


def barrier():
    if dist.is_available() and dist.is_initialized():
        dist.barrier()


def cleanup_distributed():
    if dist.is_available() and dist.is_initialized():
        dist.destroy_process_group()


setup_distributed()

random.seed(cfg['seed'] + get_rank())
np.random.seed(cfg['seed'] + get_rank())
torch.manual_seed(cfg['seed'] + get_rank())

if torch.cuda.is_available():
    device = torch.device(f'cuda:{get_local_rank()}') if is_dist_enabled() else torch.device('cuda')
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

if is_main_process():
    print('device:', device)
    if torch.cuda.is_available():
        print('gpu:', torch.cuda.get_device_name(0))
    print('dataset exists:', DATA_TRAIN.exists(), DATA_VAL.exists(), DATA_TEST.exists())
    print('resume_ckpt exists:', RESUME_CKPT.exists(), RESUME_CKPT)

with open(RUN_DIR / 'config.json', 'w') as f:
    json.dump(json.loads(json.dumps(cfg, default=str)), f, indent=2)


In [ ]:
CAMERA_NAMES = [
    '/camera/inner/frontal/middle',
    '/camera/inner/frontal/far',
    '/side/left/forward',
    '/side/right/forward',
]
INTRINSICS_NAMES = [n + '/intrinsic_params' for n in CAMERA_NAMES]
CAR2CAM_NAMES = [n + '/car_to_cam' for n in CAMERA_NAMES]
GT_NAME = 'gt_occupancy_grid'

BEV_H, BEV_W = 188, 126
BEV_RES = 0.8
X_RANGE = (0.0, BEV_H * BEV_RES)
Y_RANGE = (-BEV_W * BEV_RES / 2, BEV_W * BEV_RES / 2)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def kaggle_safe_name(name: str) -> str:
    return name.replace(':', '_')


def resolve_info_path(base_dir: Path, p) -> Path:
    p = Path(str(p))
    candidates = []

    candidates.append(p)
    candidates.append(base_dir / p)
    candidates.append(base_dir.parent / p)

    parts = list(p.parts)
    for anchor in [
        'autonomy_yandex_dataset_train',
        'autonomy_yandex_dataset_val',
        'autonomy_yandex_dataset_test',
        'autonomy_yandex_dataset_train_kaggle_safe',
        'autonomy_yandex_dataset_val_kaggle_safe',
        'autonomy_yandex_dataset_test_kaggle_safe',
    ]:
        if anchor in parts:
            i = parts.index(anchor)
            rel = Path(*parts[i + 1:])
            candidates.append(base_dir / rel)
            candidates.append(base_dir.parent / rel)
            safe_rel = Path(*[kaggle_safe_name(x) for x in rel.parts])
            candidates.append(base_dir / safe_rel)
            candidates.append(base_dir.parent / safe_rel)

    safe_p = Path(*[kaggle_safe_name(x) for x in parts])
    candidates.append(base_dir / safe_p)
    candidates.append(base_dir.parent / safe_p)

    seen = set()
    for cand in candidates:
        cand = Path(cand)
        key = str(cand)
        if key in seen:
            continue
        seen.add(key)
        if cand.exists():
            return cand

    raise FileNotFoundError(f'Could not resolve path: {p} from base_dir={base_dir}')


def load_info_with_root(data_dir: Path, split_name: str) -> pd.DataFrame:
    df = pd.read_csv(data_dir / 'info.csv', index_col=0).reset_index(drop=True).copy()
    df['__data_root'] = str(data_dir)
    df['__source_split'] = split_name
    return df


def remap_kaggle_paths(df: pd.DataFrame, train_dir: Path, val_dir: Path, test_dir: Path) -> pd.DataFrame:
    df = df.copy()
    path_cols = [*CAMERA_NAMES, *INTRINSICS_NAMES, *CAR2CAM_NAMES, GT_NAME]

    def pick_root(split: str) -> Path:
        if split == 'train':
            return train_dir
        if split == 'val':
            return val_dir
        if split == 'test':
            return test_dir
        return train_dir

    def rewrite_path(p, split: str):
        if pd.isna(p):
            return p
        p = str(p)
        pp = Path(p)
        if pp.exists():
            return str(pp)

        root = pick_root(str(split))
        parts = list(Path(p).parts)

        for anchor in [
            'autonomy_yandex_dataset_train',
            'autonomy_yandex_dataset_val',
            'autonomy_yandex_dataset_test',
            'autonomy_yandex_dataset_train_kaggle_safe',
            'autonomy_yandex_dataset_val_kaggle_safe',
            'autonomy_yandex_dataset_test_kaggle_safe',
        ]:
            if anchor in parts:
                i = parts.index(anchor)
                rel = Path(*parts[i + 1:])
                cand = root / rel
                if cand.exists():
                    return str(cand)
                safe_rel = Path(*[kaggle_safe_name(x) for x in rel.parts])
                cand = root / safe_rel
                if cand.exists():
                    return str(cand)

        safe_parts = [kaggle_safe_name(x) for x in parts]
        safe_p = Path(*safe_parts)
        cand = root / safe_p
        if cand.exists():
            return str(cand)

        cand = root / kaggle_safe_name(Path(p).name)
        if cand.exists():
            return str(cand)

        return p

    if '__source_split' in df.columns:
        df['__data_root'] = df['__source_split'].map({
            'train': str(train_dir),
            'val': str(val_dir),
            'test': str(test_dir),
        }).fillna(str(train_dir))
    else:
        df['__data_root'] = str(train_dir)

    for col in path_cols:
        if col in df.columns:
            df[col] = [
                rewrite_path(p, split)
                for p, split in zip(df[col], df.get('__source_split', pd.Series(['train'] * len(df))))
            ]

    return df


def resolve_row_path(row: pd.Series, key: str) -> Path:
    return resolve_info_path(Path(row['__data_root']), row[key])


In [ ]:
def compute_gt_stats(info_df: pd.DataFrame, cache_csv: Path | None = None) -> pd.DataFrame:
    if cache_csv is not None and cache_csv.exists():
        stats = pd.read_csv(cache_csv)
        if len(stats) == len(info_df):
            return stats

    rows = []
    for i, row in tqdm(info_df.iterrows(), total=len(info_df), desc='GT stats'):
        gt = np.load(resolve_row_path(row, GT_NAME)).squeeze()
        gt = np.where(gt < 0, 255, gt)
        valid = (gt != 255)
        pos = (gt == 1)
        rows.append({
            '__row_id': int(i),
            'coverage': float(valid.mean()),
            'valid_count': int(valid.sum()),
            'pos_count': int(pos.sum()),
        })
    stats = pd.DataFrame(rows)
    if cache_csv is not None:
        stats.to_csv(cache_csv, index=False)
    return stats


def build_img_hash(path: Path) -> np.ndarray:
    img = Image.open(path).convert('L').resize((32, 32), Image.BILINEAR)
    return np.asarray(img, dtype=np.float32) / 255.0


def smart_deduplicate(info_df: pd.DataFrame, mae_thr: float = 0.02,
                      camera_name: str = '/camera/inner/frontal/middle'):
    info_sorted = info_df.sort_values(['rover', 'ride_date', 'message_ts']).reset_index(drop=False)
    hash_cache = {}
    keep_row_ids = []
    duplicate_groups = []

    def get_hash_for_row(orig_row_idx: int, row: pd.Series):
        if orig_row_idx not in hash_cache:
            hash_cache[orig_row_idx] = build_img_hash(resolve_row_path(row, camera_name))
        return hash_cache[orig_row_idx]

    def flush_cluster(cluster):
        if not cluster:
            return
        if len(cluster) == 1:
            keep_row_ids.append(cluster[0]['orig_row_idx'])
            return
        best = sorted(
            cluster,
            key=lambda x: (-x['pos_count'], -x['valid_count'], x['message_ts'])
        )[0]
        keep_row_ids.append(best['orig_row_idx'])
        duplicate_groups.append({
            'kept_row_id': int(best['orig_row_idx']),
            'group_size': int(len(cluster)),
            'members': [int(x['orig_row_idx']) for x in cluster],
        })

    for (_, _), sub in tqdm(info_sorted.groupby(['rover', 'ride_date'], sort=False), desc='Smart dedup groups'):
        records = []
        for _, r in sub.iterrows():
            orig_idx = int(r['index'])
            records.append({
                'orig_row_idx': orig_idx,
                'row': info_df.iloc[orig_idx],
                'pos_count': int(info_df.iloc[orig_idx]['pos_count']),
                'valid_count': int(info_df.iloc[orig_idx]['valid_count']),
                'message_ts': str(info_df.iloc[orig_idx].get('message_ts', '')),
            })
        if not records:
            continue

        cluster = [records[0]]
        prev_hash = get_hash_for_row(records[0]['orig_row_idx'], records[0]['row'])
        for rec in records[1:]:
            cur_hash = get_hash_for_row(rec['orig_row_idx'], rec['row'])
            mae = float(np.mean(np.abs(cur_hash - prev_hash)))
            if mae < mae_thr:
                cluster.append(rec)
            else:
                flush_cluster(cluster)
                cluster = [rec]
            prev_hash = cur_hash
        flush_cluster(cluster)

    keep_row_ids = sorted(set(keep_row_ids))
    cleaned = info_df.iloc[keep_row_ids].reset_index(drop=True).copy()
    dup_df = pd.DataFrame(duplicate_groups)
    return cleaned, dup_df


def clean_merged_info(train_dir: Path, val_dir: Path, cache_dir: Path,
                      mae_thr: float = 0.02, dedup_camera: str = '/camera/inner/frontal/middle',
                      use_cache: bool = True):
    merged_cache = cache_dir / 'merged_cleaned.csv'
    stats_cache = cache_dir / 'merged_gt_stats.csv'
    dup_cache = cache_dir / 'dedup_report.csv'
    summary_path = cache_dir / 'clean_summary.json'

    if use_cache and merged_cache.exists() and summary_path.exists():
        info = pd.read_csv(merged_cache)
        dup_df = pd.read_csv(dup_cache) if dup_cache.exists() else pd.DataFrame()
        summary = json.loads(summary_path.read_text())
        return info, dup_df, summary

    train_info = load_info_with_root(train_dir, 'train')
    val_info = load_info_with_root(val_dir, 'val')
    merged = pd.concat([train_info, val_info], ignore_index=True)

    stats = compute_gt_stats(merged, cache_csv=stats_cache)
    merged = merged.join(stats[['coverage', 'valid_count', 'pos_count']])

    before = len(merged)
    merged = merged[merged['pos_count'] > 0].reset_index(drop=True).copy()
    after_empty = len(merged)

    deduped, dup_df = smart_deduplicate(
        merged,
        mae_thr=mae_thr,
        camera_name=dedup_camera,
    )
    after_dedup = len(deduped)

    deduped.to_csv(merged_cache, index=False)
    dup_df.to_csv(dup_cache, index=False)
    summary = {
        'merged_before_clean': int(before),
        'removed_empty_gt': int(before - after_empty),
        'after_empty_filter': int(after_empty),
        'removed_by_dedup': int(after_empty - after_dedup),
        'clean_total': int(after_dedup),
        'dedup_groups': int(len(dup_df)),
    }
    summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False))
    return deduped, dup_df, summary


In [ ]:
def make_test_matched_split_target(info_df: pd.DataFrame,
                                   test_info_csv: Path,
                                   target_val_size: int = 200,
                                   group_cols=('rover', 'ride_date'),
                                   seed: int = 42,
                                   cache_path: Path | None = None):
    if cache_path is not None and cache_path.exists():
        d = np.load(cache_path)
        return d['train_idx'].tolist(), d['val_idx'].tolist()

    rng = np.random.RandomState(seed)
    info = info_df.reset_index(drop=True).copy()
    test_info = pd.read_csv(test_info_csv, index_col=0).reset_index(drop=True)
    test_counts = Counter(test_info['rover'])
    test_total = max(sum(test_counts.values()), 1)

    grouped = info.groupby(list(group_cols)).groups
    group_rows = []
    for key, idxs in grouped.items():
        rover = key[0]
        group_rows.append({
            'group_key': key,
            'rover': rover,
            'idxs': list(sorted(int(i) for i in idxs)),
            'size': int(len(idxs)),
        })
    groups_df = pd.DataFrame(group_rows)
    groups_df['test_weight'] = groups_df['rover'].map(lambda r: test_counts.get(r, 0) / test_total)
    groups_df = groups_df[groups_df['test_weight'] > 0].reset_index(drop=True)

    selected = set()

    def choose_groups(candidate_df: pd.DataFrame, target_count: int):
        candidate_df = candidate_df.sample(frac=1.0, random_state=rng.randint(0, 10**9)).reset_index(drop=True)
        chosen = []
        total = 0
        remaining = candidate_df.to_dict('records')
        while remaining and total < target_count:
            residual = target_count - total
            remaining.sort(key=lambda x: (abs(x['size'] - residual), x['size']))
            g = remaining.pop(0)
            chosen.append(g)
            total += g['size']
            if residual <= 0:
                break
        return chosen

    rover_order = [r for r, _ in test_counts.most_common() if r in set(groups_df['rover'])]
    selected_rows = []
    for rover in rover_order:
        rover_groups = groups_df[groups_df['rover'] == rover]
        if len(rover_groups) == 0:
            continue
        target = max(1, int(round(target_val_size * test_counts[rover] / test_total)))
        chosen = choose_groups(rover_groups, target)
        for g in chosen:
            if g['group_key'] not in selected:
                selected.add(g['group_key'])
                selected_rows.append(g)

    current = sum(g['size'] for g in selected_rows)
    remaining_df = groups_df[~groups_df['group_key'].isin(selected)].copy()
    while current < target_val_size and len(remaining_df) > 0:
        residual = target_val_size - current
        remaining_df = remaining_df.sample(frac=1.0, random_state=rng.randint(0, 10**9)).reset_index(drop=True)
        remaining_df = remaining_df.sort_values(['test_weight', 'size'], ascending=[False, True]).reset_index(drop=True)
        best_idx = min(range(len(remaining_df)), key=lambda i: (abs(int(remaining_df.iloc[i]['size']) - residual), -float(remaining_df.iloc[i]['test_weight'])))
        g = remaining_df.iloc[best_idx].to_dict()
        selected.add(g['group_key'])
        selected_rows.append(g)
        current += int(g['size'])
        remaining_df = remaining_df[remaining_df['group_key'] != g['group_key']].reset_index(drop=True)

    selected_df = pd.DataFrame(selected_rows)
    while len(selected_df) > 1 and selected_df['size'].sum() > target_val_size + 20:
        overflow = int(selected_df['size'].sum() - target_val_size)
        rover_group_counts = selected_df.groupby('rover').size().to_dict()
        candidate_ids = []
        for i, row in selected_df.iterrows():
            if rover_group_counts.get(row['rover'], 0) > 1:
                candidate_ids.append(i)
        if not candidate_ids:
            break
        drop_i = min(candidate_ids, key=lambda i: abs(int(selected_df.loc[i, 'size']) - overflow))
        selected_df = selected_df.drop(index=drop_i).reset_index(drop=True)

    val_groups = set(selected_df['group_key'].tolist())
    train_idx, val_idx = [], []
    for i, row in info.iterrows():
        key = tuple(row[c] for c in group_cols)
        if key in val_groups:
            val_idx.append(i)
        else:
            train_idx.append(i)

    if cache_path is not None:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        np.savez(cache_path, train_idx=np.array(train_idx), val_idx=np.array(val_idx))
    return train_idx, val_idx


def build_rover_vocab_from_train(train_df: pd.DataFrame,
                                 min_count: int = 30,
                                 topk: int = 25):
    counts = train_df['rover'].value_counts()
    eligible = counts[counts >= min_count]
    top = eligible.head(topk)
    vocab = {'__other__': 0}
    for i, rover in enumerate(top.index.tolist(), start=1):
        vocab[rover] = i
    stats_df = pd.DataFrame({
        'rover': counts.index,
        'count': counts.values,
        'embedding_id': [vocab.get(r, 0) for r in counts.index],
        'bucket': ['unique' if r in vocab and r != '__other__' else 'other' for r in counts.index],
    })
    return vocab, stats_df


def encode_rover(rover_name: str, rover_vocab: dict) -> int:
    return int(rover_vocab.get(rover_name, 0))


In [ ]:
clean_info, dedup_report, clean_summary = clean_merged_info(
    DATA_TRAIN,
    DATA_VAL,
    cache_dir=CACHE_DIR,
    mae_thr=cfg['mae_dedup_thr'],
    dedup_camera=cfg['dedup_camera'],
    use_cache=cfg['use_clean_cache'],
)
clean_info = remap_kaggle_paths(clean_info, DATA_TRAIN, DATA_VAL, DATA_TEST)
print(json.dumps(clean_summary, indent=2, ensure_ascii=False))
display(clean_info.head(3))
if len(dedup_report):
    display(dedup_report.head(10))

train_idx, val_idx = make_test_matched_split_target(
    clean_info,
    DATA_TEST / 'info.csv',
    target_val_size=cfg['val_target_size'],
    seed=cfg['seed'],
    cache_path=CACHE_DIR / 'test_matched_val200_split.npz',
)
print('train_idx:', len(train_idx), 'val_idx:', len(val_idx))

train_info = clean_info.iloc[train_idx].reset_index(drop=True).copy()
val_info = clean_info.iloc[val_idx].reset_index(drop=True).copy()
train_info = remap_kaggle_paths(train_info, DATA_TRAIN, DATA_VAL, DATA_TEST)
val_info = remap_kaggle_paths(val_info, DATA_TRAIN, DATA_VAL, DATA_TEST)

rover_vocab, rover_stats = build_rover_vocab_from_train(
    train_info,
    min_count=cfg['min_rover_count'],
    topk=cfg['topk_rovers'],
)
rover_stats.to_csv(RUN_DIR / 'rover_embedding_stats.csv', index=False)
print('num rover classes including Other:', len(rover_vocab))
print('top rover mapping:', rover_vocab)
display(rover_stats.head(30))

if is_main_process() and len(val_info):
    row = val_info.iloc[0]
    for key in [CAMERA_NAMES[0], INTRINSICS_NAMES[0], CAR2CAM_NAMES[0], GT_NAME]:
        path = resolve_info_path(Path(row['__data_root']), row[key])
        print(key, path, path.exists())


In [ ]:
class BEVDatasetV4Clean(Dataset):
    def __init__(self, info_df: pd.DataFrame, mode: str = 'train',
                 img_hw=(384, 704), aug: bool = False,
                 rover_vocab: dict | None = None):
        self.info = info_df.reset_index(drop=True).copy()
        self.mode = mode
        self.img_hw = img_hw
        self.aug = aug and (mode == 'train')
        self.rover_vocab = rover_vocab or {'__other__': 0}
        self.normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
        self.color_jitter = transforms.ColorJitter(
            brightness=0.10,
            contrast=0.10,
            saturation=0.06,
            hue=0.02,
        )

    def __len__(self):
        return len(self.info)

    def _resolve_path(self, row: pd.Series, key: str) -> Path:
        return resolve_info_path(Path(row['__data_root']), row[key])

    def _load_one_camera(self, row: pd.Series, cam_key: str, intr_key: str, c2c_key: str,
                         scale_aug: float = 1.0):
        img = Image.open(self._resolve_path(row, cam_key)).convert('RGB')
        intr_path = self._resolve_path(row, intr_key)
        car2cam_path = self._resolve_path(row, c2c_key)
        src_W, src_H = img.size
        H_t, W_t = self.img_hw

        s = min(W_t / src_W, H_t / src_H)
        new_W, new_H = int(round(src_W * s)), int(round(src_H * s))
        img_resized = img.resize((new_W, new_H), Image.BILINEAR)
        canvas = Image.new('RGB', (W_t, H_t), (0, 0, 0))
        pad_x = (W_t - new_W) // 2
        pad_y = (H_t - new_H) // 2
        canvas.paste(img_resized, (pad_x, pad_y))

        extra_s = 1.0
        extra_dx = 0
        extra_dy = 0
        if scale_aug > 1.0:
            sH, sW = int(round(H_t * scale_aug)), int(round(W_t * scale_aug))
            canvas = canvas.resize((sW, sH), Image.BILINEAR)
            extra_dx = random.randint(0, sW - W_t)
            extra_dy = random.randint(0, sH - H_t)
            canvas = canvas.crop((extra_dx, extra_dy, extra_dx + W_t, extra_dy + H_t))
            extra_s = scale_aug

        if self.aug:
            if random.random() < 0.70:
                canvas = self.color_jitter(canvas)
            if random.random() < 0.15:
                canvas = canvas.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.1, 0.8)))

        arr = np.array(canvas)
        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)
        img_t = torch.from_numpy(arr).permute(2, 0, 1).float() / 255.0
        img_t = self.normalize(img_t)

        intr_full = np.load(intr_path)
        K = intr_full[:, :3].copy().astype(np.float32)
        K[0, 0] *= s; K[0, 2] *= s
        K[1, 1] *= s; K[1, 2] *= s
        K[0, 2] += pad_x
        K[1, 2] += pad_y
        K[0, 0] *= extra_s; K[0, 2] *= extra_s
        K[1, 1] *= extra_s; K[1, 2] *= extra_s
        K[0, 2] -= extra_dx
        K[1, 2] -= extra_dy

        car2cam = np.load(car2cam_path).astype(np.float32)
        return img_t, K, car2cam

    def _load_sample(self, idx: int):
        row = self.info.iloc[idx]
        scale_aug = random.uniform(1.0, 1.15) if self.aug else 1.0

        imgs, Ks, M = [], [], []
        for cam, intr, c2c in zip(CAMERA_NAMES, INTRINSICS_NAMES, CAR2CAM_NAMES):
            img_t, K, c = self._load_one_camera(row, cam, intr, c2c, scale_aug=scale_aug)
            imgs.append(img_t)
            Ks.append(torch.from_numpy(K))
            M.append(torch.from_numpy(c))

        out = {
            'images': torch.stack(imgs, dim=0),
            'intrinsics': torch.stack(Ks, dim=0),
            'car2cams': torch.stack(M, dim=0),
            'rover_id': torch.tensor(encode_rover(row.get('rover', '__other__'), self.rover_vocab), dtype=torch.long),
            'info_idx': idx,
        }
        if self.mode == 'test':
            return out

        gt = np.load(self._resolve_path(row, GT_NAME)).squeeze()
        gt = np.where(gt < 0, 255, gt).astype(np.int64)
        out['gt'] = torch.from_numpy(gt).unsqueeze(0)
        return out

    def __getitem__(self, idx: int):
        max_tries = 5
        last_err = None
        for k in range(max_tries):
            try_idx = (idx + k) % len(self.info)
            try:
                return self._load_sample(try_idx)
            except (OSError, ValueError, FileNotFoundError) as e:
                last_err = e
                continue
        raise RuntimeError(f'Failed to load sample after {max_tries} tries from idx={idx}: {last_err}')


In [ ]:
def gn_groups(channels: int, requested: int = 8) -> int:
    g = min(requested, channels)
    while channels % g != 0 and g > 1:
        g -= 1
    return max(g, 1)


class ConvGNAct(nn.Module):
    def __init__(self, in_c, out_c, k=3, s=1, p=1, groups=8, act=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_c, out_c, k, stride=s, padding=p, bias=False),
            nn.GroupNorm(gn_groups(out_c, groups), out_c),
        ]
        if act:
            layers.append(nn.SiLU(inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

class ResidualBlock2d(nn.Module):
    def __init__(self, in_c, out_c, stride=1, groups=8):
        super().__init__()
        self.conv1 = ConvGNAct(in_c, out_c, k=3, s=stride, p=1, groups=groups, act=True)
        self.conv2 = ConvGNAct(out_c, out_c, k=3, s=1, p=1, groups=groups, act=False)
        if stride != 1 or in_c != out_c:
            self.skip = ConvGNAct(in_c, out_c, k=1, s=stride, p=0, groups=groups, act=False)
        else:
            self.skip = nn.Identity()
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        return self.act(self.conv2(self.conv1(x)) + self.skip(x))

class ASPP2d(nn.Module):
    def __init__(self, in_c, out_c, rates=(1, 3, 6), groups=8):
        super().__init__()
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=r, dilation=r, bias=False),
                nn.GroupNorm(gn_groups(out_c, groups), out_c),
                nn.SiLU(inplace=True),
            )
            for r in rates
        ])
        self.proj = ConvGNAct(out_c * len(rates), out_c, k=1, s=1, p=0, groups=groups, act=True)

    def forward(self, x):
        xs = [b(x) for b in self.branches]
        return self.proj(torch.cat(xs, dim=1))

class _DINOv2MultiScaleBackbone(nn.Module):
    def __init__(self,
                 hub_repo: str = 'facebookresearch/dinov2',
                 backbone_name: str = 'dinov2_vitb14',
                 out_dim: int = 768,
                 patch_size: int = 14,
                 tap_layers=(2, 5, 8, 11),
                 neck_dim: int = 128,
                 groups: int = 8):
        super().__init__()
        self.hub_repo = hub_repo
        self.backbone_name = backbone_name
        self.out_dim = out_dim
        self.patch_size = patch_size
        self.tap_layers = tuple(tap_layers)
        self.neck_dim = neck_dim

        self.vit = self._load_hub_model()
        self.laterals = nn.ModuleList([nn.Conv2d(out_dim, neck_dim, 1) for _ in self.tap_layers])
        self.fuse = nn.Sequential(
            ConvGNAct(len(self.tap_layers) * neck_dim, neck_dim, k=3, s=1, p=1, groups=groups, act=True),
            ConvGNAct(neck_dim, neck_dim, k=3, s=1, p=1, groups=groups, act=True),
        )
        self.down1 = ConvGNAct(neck_dim, neck_dim, k=3, s=2, p=1, groups=groups, act=True)
        self.down2 = ConvGNAct(neck_dim, neck_dim, k=3, s=2, p=1, groups=groups, act=True)
        self.neck_out = ConvGNAct(neck_dim * 3, neck_dim, k=1, s=1, p=0, groups=groups, act=True)

    def _load_hub_model(self):
        last_err = None
        attempts = [
            dict(repo_or_dir=self.hub_repo, model=self.backbone_name),
            dict(repo_or_dir=self.hub_repo, model=self.backbone_name, pretrained=True),
            dict(repo_or_dir=self.hub_repo, model=self.backbone_name, source='github'),
            dict(repo_or_dir=self.hub_repo, model=self.backbone_name, source='github', pretrained=True),
        ]
        for kwargs in attempts:
            try:
                return torch.hub.load(**kwargs)
            except Exception as e:
                last_err = e
        raise RuntimeError(
            f'Failed to load DINOv2 backbone {self.backbone_name} from {self.hub_repo}. '
            f'Last error: {last_err}'
        )

    def _reshape_tokens(self, tokens: torch.Tensor, H: int, W: int) -> torch.Tensor:
        B = tokens.shape[0]
        expected_tokens = (H // self.patch_size) * (W // self.patch_size)
        if tokens.ndim != 3:
            raise RuntimeError(f'Unexpected token shape: {tuple(tokens.shape)}')
        if tokens.shape[1] == expected_tokens + 1:
            tokens = tokens[:, 1:, :]
        elif tokens.shape[1] != expected_tokens:
            raise RuntimeError(
                f'Unexpected number of DINO tokens: got {tokens.shape[1]}, expected {expected_tokens} '
                f'for img_hw={(H, W)}'
            )
        Hp = H // self.patch_size
        Wp = W // self.patch_size
        return tokens.transpose(1, 2).reshape(B, self.out_dim, Hp, Wp).contiguous()

    def _extract_intermediate(self, x: torch.Tensor):
        H, W = x.shape[-2:]
        try:
            feats = self.vit.get_intermediate_layers(
                x,
                n=list(self.tap_layers),
                reshape=True,
                return_class_token=False,
            )
        except Exception:
            feats = self.vit.get_intermediate_layers(
                x,
                n=len(self.tap_layers),
                reshape=True,
                return_class_token=False,
            )

        out = []
        for feat in feats:
            if isinstance(feat, (tuple, list)):
                feat = feat[0]
            if feat.ndim == 3:
                feat = self._reshape_tokens(feat, H, W)
            out.append(feat)
        if len(out) != len(self.tap_layers):
            raise RuntimeError(f'Expected {len(self.tap_layers)} intermediate features, got {len(out)}')
        return out

    def forward(self, x):
        feats = self._extract_intermediate(x)
        laterals = [proj(feat) for proj, feat in zip(self.laterals, feats)]
        p0 = self.fuse(torch.cat(laterals, dim=1))
        p1 = self.down1(p0)
        p2 = self.down2(p1)
        p1_up = F.interpolate(p1, size=p0.shape[-2:], mode='bilinear', align_corners=False)
        p2_up = F.interpolate(p2, size=p0.shape[-2:], mode='bilinear', align_corners=False)
        fused = self.neck_out(torch.cat([p0, p1_up, p2_up], dim=1))
        return {
            'tap_features': feats,
            'p0': p0,
            'p1': p1,
            'p2': p2,
            'fused': fused,
        }

class LSSViewTransform2D(nn.Module):
    def __init__(self,
                 in_c: int,
                 context_c: int,
                 depth_bins: int,
                 depth_min: float,
                 depth_max: float,
                 bev_h: int,
                 bev_w: int,
                 bev_res: float,
                 x_range,
                 y_range,
                 z_min: float,
                 z_max: float,
                 groups: int = 8):
        super().__init__()
        self.context_c = context_c
        self.depth_bins = depth_bins
        self.depth_min = float(depth_min)
        self.depth_max = float(depth_max)
        self.bev_h = bev_h
        self.bev_w = bev_w
        self.bev_res = float(bev_res)
        self.x_range = x_range
        self.y_range = y_range
        self.z_min = float(z_min)
        self.z_max = float(z_max)

        self.depth_head = nn.Sequential(
            ConvGNAct(in_c, in_c, k=3, s=1, p=1, groups=groups, act=True),
            nn.Conv2d(in_c, depth_bins, 1),
        )
        self.context_head = nn.Sequential(
            ConvGNAct(in_c, in_c, k=3, s=1, p=1, groups=groups, act=True),
            nn.Conv2d(in_c, context_c, 1),
        )

    def _build_frustum(self, Hf: int, Wf: int, Hi: int, Wi: int, device, dtype):
        depths = torch.linspace(self.depth_min, self.depth_max, self.depth_bins, device=device, dtype=dtype)
        xs = (torch.arange(Wf, device=device, dtype=dtype) + 0.5) * (Wi / Wf)
        ys = (torch.arange(Hf, device=device, dtype=dtype) + 0.5) * (Hi / Hf)
        d, y, x = torch.meshgrid(depths, ys, xs, indexing='ij')
        return x, y, d

    def forward(self, feat_2d: torch.Tensor, intrinsics: torch.Tensor, car2cams: torch.Tensor, image_hw):
        B, N, C, Hf, Wf = feat_2d.shape
        Hi, Wi = image_hw

        feat_bn = feat_2d.reshape(B * N, C, Hf, Wf)
        depth_logits = self.depth_head(feat_bn).float()
        context = self.context_head(feat_bn).float()

        depth_prob = torch.softmax(depth_logits, dim=1)
        depth_prob = depth_prob.reshape(B, N, self.depth_bins, Hf, Wf)
        context = context.reshape(B, N, self.context_c, Hf, Wf)

        x_img, y_img, depth_vals = self._build_frustum(Hf, Wf, Hi, Wi, feat_2d.device, torch.float32)
        x_img = x_img.view(1, 1, self.depth_bins, Hf, Wf)
        y_img = y_img.view(1, 1, self.depth_bins, Hf, Wf)
        depth_vals = depth_vals.view(1, 1, self.depth_bins, Hf, Wf)

        intrinsics = intrinsics.float()
        car2cams = car2cams.float()
        cam2cars = torch.inverse(car2cams.reshape(B * N, 4, 4)).reshape(B, N, 4, 4)

        fx = intrinsics[..., 0, 0].view(B, N, 1, 1, 1)
        fy = intrinsics[..., 1, 1].view(B, N, 1, 1, 1)
        cx = intrinsics[..., 0, 2].view(B, N, 1, 1, 1)
        cy = intrinsics[..., 1, 2].view(B, N, 1, 1, 1)

        X = (x_img - cx) / fx * depth_vals
        Y = (y_img - cy) / fy * depth_vals
        Z = depth_vals.expand(B, N, -1, -1, -1)
        ones = torch.ones_like(Z)
        pts_cam = torch.stack([X, Y, Z, ones], dim=-1)
        pts_car = torch.einsum('bnij,bndhwj->bndhwi', cam2cars, pts_cam)

        world_x = pts_car[..., 0]
        world_y = pts_car[..., 1]
        world_z = pts_car[..., 2]

        x_idx = torch.floor((world_x - self.x_range[0]) / self.bev_res).long()
        y_idx = torch.floor((world_y - self.y_range[0]) / self.bev_res).long()
        valid = (
            (x_idx >= 0) & (x_idx < self.bev_h) &
            (y_idx >= 0) & (y_idx < self.bev_w) &
            (world_z >= self.z_min) & (world_z <= self.z_max)
        )
        linear_idx = x_idx * self.bev_w + y_idx

        feat_vol = context.unsqueeze(3) * depth_prob.unsqueeze(2)
        bev = feat_2d.new_zeros(B, self.context_c, self.bev_h * self.bev_w, dtype=torch.float32)
        counts = feat_2d.new_zeros(B, 1, self.bev_h * self.bev_w, dtype=torch.float32)

        for b in range(B):
            idx_b = linear_idx[b].reshape(-1)
            valid_b = valid[b].reshape(-1)
            if not valid_b.any():
                continue
            feat_b = feat_vol[b].permute(1, 0, 2, 3, 4).reshape(self.context_c, -1)
            idx_valid = idx_b[valid_b]
            feat_valid = feat_b[:, valid_b]
            bev[b].scatter_add_(1, idx_valid.unsqueeze(0).expand(self.context_c, -1), feat_valid)
            counts[b].scatter_add_(1, idx_valid.unsqueeze(0), torch.ones(1, idx_valid.numel(), device=feat_2d.device, dtype=torch.float32))

        bev = bev / counts.clamp(min=1.0)
        bev = bev.reshape(B, self.context_c, self.bev_h, self.bev_w)
        bev = torch.nan_to_num(bev, nan=0.0, posinf=0.0, neginf=0.0)

        debug = {
            'depth_logits': depth_logits.reshape(B, N, self.depth_bins, Hf, Wf),
            'depth_prob': depth_prob,
            'context': context,
            'bev': bev,
            'valid_ratio': valid.float().mean().item(),
        }
        return bev, debug

def load_warm_start(model: nn.Module, ckpt_path: str | Path):
    ckpt_path = Path(ckpt_path)
    if not ckpt_path.exists():
        print('warm-start checkpoint not found, starting from random init:', ckpt_path)
        return {'loaded': 0, 'skipped': 0, 'missing': None, 'unexpected': None}

    ckpt = torch.load(ckpt_path, map_location='cpu')
    state = ckpt['ema'] if 'ema' in ckpt else ckpt.get('model', ckpt)
    cur = model.state_dict()
    loadable = {}
    skipped = []
    for k, v in state.items():
        if k in cur and cur[k].shape == v.shape:
            loadable[k] = v
        else:
            skipped.append(k)
    missing, unexpected = model.load_state_dict(loadable, strict=False)
    print('loaded warm-start from', ckpt_path)
    print('  matched keys:', len(loadable))
    print('  skipped old keys:', len(skipped))
    print('  missing in new model:', len(missing))
    print('  unexpected during load:', len(unexpected))
    if len(skipped):
        print('  sample skipped:', skipped[:10])
    return {
        'loaded': len(loadable),
        'skipped': len(skipped),
        'missing': missing,
        'unexpected': unexpected,
    }

class StrongBEVEncoderDecoderHiRes(nn.Module):
    def __init__(self, in_c: int, base_c: int = 96, groups: int = 8, out_down_factor: int = 2):
        super().__init__()
        self.out_down_factor = int(out_down_factor)
        self.stem = nn.Sequential(
            ConvGNAct(in_c, base_c, k=3, s=1, p=1, groups=groups, act=True),
            ResidualBlock2d(base_c, base_c, stride=1, groups=groups),
        )
        self.down1 = nn.Sequential(
            ResidualBlock2d(base_c, base_c * 2, stride=2, groups=groups),
            ResidualBlock2d(base_c * 2, base_c * 2, stride=1, groups=groups),
        )
        self.down2 = nn.Sequential(
            ResidualBlock2d(base_c * 2, base_c * 4, stride=2, groups=groups),
            ResidualBlock2d(base_c * 4, base_c * 4, stride=1, groups=groups),
        )
        self.aspp = ASPP2d(base_c * 4, base_c * 4, rates=(1, 3, 6), groups=groups)
        self.up1 = nn.Sequential(
            ConvGNAct(base_c * 4 + base_c * 2, base_c * 2, k=3, s=1, p=1, groups=groups, act=True),
            ResidualBlock2d(base_c * 2, base_c * 2, stride=1, groups=groups),
        )
        self.up0 = nn.Sequential(
            ConvGNAct(base_c * 2 + base_c, base_c, k=3, s=1, p=1, groups=groups, act=True),
            ResidualBlock2d(base_c, base_c, stride=1, groups=groups),
        )
        self.head = nn.Conv2d(base_c, 1, 1)

    def forward_debug(self, x):
        s0 = self.stem(x)
        s1 = self.down1(s0)
        s2 = self.down2(s1)
        b = self.aspp(s2)
        u1 = self.up1(torch.cat([F.interpolate(b, size=s1.shape[-2:], mode='bilinear', align_corners=False), s1], dim=1))
        u0 = self.up0(torch.cat([F.interpolate(u1, size=s0.shape[-2:], mode='bilinear', align_corners=False), s0], dim=1))
        logits_hr = self.head(u0)
        if self.out_down_factor > 1:
            logits = F.avg_pool2d(logits_hr, kernel_size=self.out_down_factor, stride=self.out_down_factor)
        else:
            logits = logits_hr
        return {'logits_hr': logits_hr, 'logits': logits}

    def forward(self, x):
        return self.forward_debug(x)['logits']


class MultiCamBEVv62DINOv2LSSBEV2xClean(nn.Module):
    def __init__(self, num_rover_classes: int,
                 rover_emb_dim: int = 8,
                 rover_cond_dim: int = 8,
                 n_cameras: int = 4,
                 freeze_backbone: bool = False,
                 hub_repo: str = 'facebookresearch/dinov2',
                 backbone_name: str = 'dinov2_vitb14',
                 backbone_out_dim: int = 768,
                 patch_size: int = 14,
                 tap_layers=(2, 5, 8, 11),
                 neck_dim: int = 128,
                 context_dim: int = 80,
                 depth_bins: int = 24,
                 depth_min: float = 1.0,
                 depth_max: float = 80.0,
                 world_z_min: float = -2.0,
                 world_z_max: float = 4.5,
                 bev_base_channels: int = 96,
                 bev_gn_groups: int = 8,
                 bev_upsample_factor: int = 2):
        super().__init__()
        self.n_cameras = n_cameras
        self.rover_cond_dim = rover_cond_dim
        self.bev_upsample_factor = int(bev_upsample_factor)
        self.bev_h_hr = BEV_H * self.bev_upsample_factor
        self.bev_w_hr = BEV_W * self.bev_upsample_factor
        self.bev_res_hr = BEV_RES / float(self.bev_upsample_factor)

        self.backbone = _DINOv2MultiScaleBackbone(
            hub_repo=hub_repo,
            backbone_name=backbone_name,
            out_dim=backbone_out_dim,
            patch_size=patch_size,
            tap_layers=tap_layers,
            neck_dim=neck_dim,
            groups=bev_gn_groups,
        )
        if freeze_backbone:
            for p in self.backbone.vit.parameters():
                p.requires_grad = False

        self.view_transform = LSSViewTransform2D(
            in_c=neck_dim,
            context_c=context_dim,
            depth_bins=depth_bins,
            depth_min=depth_min,
            depth_max=depth_max,
            bev_h=self.bev_h_hr,
            bev_w=self.bev_w_hr,
            bev_res=self.bev_res_hr,
            x_range=X_RANGE,
            y_range=Y_RANGE,
            z_min=world_z_min,
            z_max=world_z_max,
            groups=bev_gn_groups,
        )

        self.rover_embed = nn.Embedding(num_rover_classes, rover_emb_dim)
        nn.init.normal_(self.rover_embed.weight, std=0.02)
        self.rover_mlp = nn.Sequential(
            nn.Linear(rover_emb_dim, 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, rover_cond_dim),
            nn.ReLU(inplace=True),
        )

        self.bev_decoder = StrongBEVEncoderDecoderHiRes(
            in_c=context_dim + rover_cond_dim,
            base_c=bev_base_channels,
            groups=bev_gn_groups,
            out_down_factor=self.bev_upsample_factor,
        )

    def forward_debug(self, images, intrinsics, car2cams, rover_ids):
        B, N, C, H, W = images.shape
        assert N == self.n_cameras
        x = images.reshape(B * N, C, H, W)
        back = self.backbone(x)
        fused = back['fused'].reshape(B, N, back['fused'].shape[1], back['fused'].shape[2], back['fused'].shape[3])
        bev_hr, vt_debug = self.view_transform(fused, intrinsics, car2cams, image_hw=(H, W))
        rover_feat = self.rover_mlp(self.rover_embed(rover_ids)).view(B, self.rover_cond_dim, 1, 1)
        rover_map = rover_feat.expand(-1, -1, self.bev_h_hr, self.bev_w_hr)
        dec_debug = self.bev_decoder.forward_debug(torch.cat([bev_hr, rover_map], dim=1))
        return {
            'tap_features': back['tap_features'],
            'image_fused': fused,
            'depth_logits': vt_debug['depth_logits'],
            'depth_prob': vt_debug['depth_prob'],
            'bev_raw_hr': vt_debug['bev'],
            'valid_ratio': vt_debug['valid_ratio'],
            'logits_hr': dec_debug['logits_hr'],
            'logits': dec_debug['logits'],
        }

    def forward(self, images, intrinsics, car2cams, rover_ids):
        dbg = self.forward_debug(images, intrinsics, car2cams, rover_ids)
        logits = torch.nan_to_num(dbg['logits'], nan=0.0, posinf=0.0, neginf=0.0)
        return logits


def load_hr_resume_state(core_model, ema_model, run_dir: Path):
    resume_ckpt = Path(cfg.get('resume_ckpt', ''))
    if not cfg.get('resume_training', False) or not resume_ckpt.exists():
        return {
            'enabled': False,
            'start_epoch': 0,
            'best_iou': -1.0,
            'best_ema_iou': -1.0,
            'log': [],
            'elapsed_minutes': 0.0,
        }

    ckpt = torch.load(resume_ckpt, map_location='cpu')
    core_model.load_state_dict(ckpt['model'], strict=True)
    if 'ema' in ckpt:
        ema_model.load_state_dict(ckpt['ema'], strict=True)
    else:
        ema_model.load_state_dict(ckpt['model'], strict=True)

    start_epoch = int(ckpt.get('epoch', -1)) + 1
    log_path = resume_ckpt.parent / 'log.csv'
    log_rows = []
    elapsed_minutes = 0.0
    if log_path.exists():
        log_rows = pd.read_csv(log_path).to_dict('records')
        if len(log_rows):
            elapsed_minutes = float(log_rows[-1].get('minutes', 0.0) or 0.0)

    best_iou = float(ckpt.get('best_iou', -1.0))
    best_ema_iou = float(ckpt.get('best_ema_iou', -1.0))
    print('high-res resumed from', resume_ckpt)
    print('  strict model/ema load succeeded')
    print('  start_epoch:', start_epoch)

    return {
        'enabled': True,
        'start_epoch': start_epoch,
        'best_iou': best_iou,
        'best_ema_iou': best_ema_iou,
        'log': log_rows,
        'elapsed_minutes': elapsed_minutes,
    }


In [ ]:
def _lovasz_grad(gt_sorted: torch.Tensor) -> torch.Tensor:
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1.0 - gt_sorted).float().cumsum(0)
    jaccard = 1.0 - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard


def lovasz_hinge_flat(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    if labels.numel() == 0:
        return logits.sum() * 0.0
    signs = 2.0 * labels.float() - 1.0
    errors = 1.0 - logits * signs
    errors_sorted, perm = torch.sort(errors, dim=0, descending=True)
    gt_sorted = labels[perm]
    grad = _lovasz_grad(gt_sorted)
    return torch.dot(F.relu(errors_sorted), grad)


class CompoundLossV2(nn.Module):
    def __init__(self, pos_weight: float = 5.0,
                 weight_bce: float = 0.5,
                 weight_dice: float = 0.3,
                 weight_lovasz: float = 0.2,
                 ignore_value: int = 255):
        super().__init__()
        self.register_buffer('pos_weight', torch.tensor([pos_weight]))
        self.weight_bce = weight_bce
        self.weight_dice = weight_dice
        self.weight_lovasz = weight_lovasz
        self.ignore_value = ignore_value

    def forward(self, logits: torch.Tensor, gt: torch.Tensor):
        logits = logits.float()
        gt = gt.long()
        valid = (gt != self.ignore_value)
        gt_f = (gt == 1).float()

        bce_per = F.binary_cross_entropy_with_logits(logits, gt_f, pos_weight=self.pos_weight, reduction='none')
        bce = (bce_per * valid.float()).sum() / valid.float().sum().clamp(min=1.0)

        prob = torch.sigmoid(logits) * valid.float()
        gt_d = gt_f * valid.float()
        inter = (prob * gt_d).sum(dim=(1, 2, 3))
        denom = prob.sum(dim=(1, 2, 3)) + gt_d.sum(dim=(1, 2, 3))
        dice = (1.0 - (2 * inter + 1) / (denom + 1)).mean()

        lov_logits = logits[valid]
        lov_gt = gt_f[valid]
        lov = lovasz_hinge_flat(lov_logits, lov_gt) if lov_logits.numel() > 0 else logits.sum() * 0.0

        total = self.weight_bce * bce + self.weight_dice * dice + self.weight_lovasz * lov
        parts = {'bce': float(bce.item()), 'dice': float(dice.item()), 'lovasz': float(lov.item())}
        return total, parts


def unwrap_model(model):
    return model.module if hasattr(model, 'module') else model


@torch.no_grad()
def iou_binary_batch(logits: torch.Tensor, gt: torch.Tensor, threshold: float = 0.5, ignore_value: int = 255):
    probs = torch.sigmoid(logits)
    pred = probs > threshold
    valid = gt != ignore_value
    gt_b = (gt == 1) & valid
    pred = pred & valid
    inter = (pred & gt_b).sum().item()
    union = (pred | gt_b).sum().item()
    return inter, union


@torch.no_grad()
def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


@torch.inference_mode()
def evaluate_iou(model, loader, threshold=0.5, desc='val@0.5'):
    model.eval()
    inter = 0
    union = 0
    pbar = tqdm(loader, desc=desc, leave=False)
    for batch in pbar:
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt = batch['gt'].to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda' and cfg['use_amp'])):
            logits = model(images, intr, c2c, rover_id)
        logits = logits.float()
        i, u = iou_binary_batch(logits, gt, threshold=threshold)
        inter += i
        union += u
        pbar.set_postfix(iou=f'{inter / max(union, 1):.4f}')
    return inter / max(union, 1)


In [ ]:
ds_train_warm = BEVDatasetV4Clean(train_info, mode='train', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab)
ds_train_aug = BEVDatasetV4Clean(train_info, mode='train', img_hw=cfg['img_hw'], aug=True, rover_vocab=rover_vocab)
ds_val = BEVDatasetV4Clean(val_info, mode='val', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab)

train_sampler_warm = None
train_sampler_aug = None
if is_dist_enabled():
    train_sampler_warm = DistributedSampler(ds_train_warm, num_replicas=get_world_size(), rank=get_rank(), shuffle=True, drop_last=True)
    train_sampler_aug = DistributedSampler(ds_train_aug, num_replicas=get_world_size(), rank=get_rank(), shuffle=True, drop_last=True)

loader_warm = DataLoader(ds_train_warm, batch_size=cfg['batch_size'], shuffle=(train_sampler_warm is None), sampler=train_sampler_warm, num_workers=cfg['num_workers'], pin_memory=(device.type == 'cuda'), drop_last=True)
loader_train = DataLoader(ds_train_aug, batch_size=cfg['batch_size'], shuffle=(train_sampler_aug is None), sampler=train_sampler_aug, num_workers=cfg['num_workers'], pin_memory=(device.type == 'cuda'), drop_last=True)
loader_val = None
if is_main_process():
    loader_val = DataLoader(ds_val, batch_size=cfg['val_batch_size'], shuffle=False, num_workers=cfg['val_num_workers'], pin_memory=(device.type == 'cuda'))
    print('loader_train batch_size:', loader_train.batch_size, '| workers:', loader_train.num_workers)

base_model = MultiCamBEVv62DINOv2LSSBEV2xClean(
    num_rover_classes=len(rover_vocab),
    rover_emb_dim=cfg['rover_emb_dim'],
    rover_cond_dim=cfg['rover_cond_dim'],
    freeze_backbone=False,
    hub_repo=cfg['hub_repo'],
    backbone_name=cfg['backbone_name'],
    backbone_out_dim=cfg['backbone_out_dim'],
    patch_size=cfg['patch_size'],
    tap_layers=cfg['tap_layers'],
    neck_dim=cfg['neck_dim'],
    context_dim=cfg['context_dim'],
    depth_bins=cfg['depth_bins'],
    depth_min=cfg['depth_min'],
    depth_max=cfg['depth_max'],
    world_z_min=cfg['world_z_min'],
    world_z_max=cfg['world_z_max'],
    bev_base_channels=cfg['bev_base_channels'],
    bev_gn_groups=cfg['bev_gn_groups'],
    bev_upsample_factor=cfg['bev_upsample_factor'],
).to(device)

def freeze_all_backbone(model):
    core = unwrap_model(model)
    for p in core.backbone.vit.parameters():
        p.requires_grad = False

def unfreeze_last_blocks(model, n_last_blocks=2):
    core = unwrap_model(model)
    freeze_all_backbone(core)
    if hasattr(core.backbone.vit, 'blocks'):
        for blk in core.backbone.vit.blocks[-n_last_blocks:]:
            for p in blk.parameters():
                p.requires_grad = True
    if hasattr(core.backbone.vit, 'norm'):
        for p in core.backbone.vit.norm.parameters():
            p.requires_grad = True

freeze_all_backbone(base_model)

if is_dist_enabled():
    model = DDP(base_model, device_ids=[get_local_rank()], output_device=get_local_rank(), broadcast_buffers=False, find_unused_parameters=False)
else:
    model = base_model

core_model = unwrap_model(model)
backbone_params = []
embed_params = []
image_neck_params = []
other_params = []
for name, p in core_model.named_parameters():
    if name.startswith('backbone.vit.'):
        backbone_params.append(p)
    elif name.startswith('rover_embed.'):
        embed_params.append(p)
    elif name.startswith('backbone.laterals.') or name.startswith('backbone.fuse.') or name.startswith('backbone.down') or name.startswith('backbone.neck_out.'):
        image_neck_params.append(p)
    else:
        other_params.append(p)

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': cfg['lr_backbone'], 'weight_decay': cfg['weight_decay']},
    {'params': image_neck_params, 'lr': cfg['lr_image_neck'], 'weight_decay': cfg['weight_decay']},
    {'params': other_params, 'lr': cfg['lr_lss_bev'], 'weight_decay': cfg['weight_decay']},
    {'params': embed_params, 'lr': cfg['lr_lss_bev'], 'weight_decay': cfg['embed_weight_decay']},
])
remaining_epochs = max(cfg['epochs'] - max(0, int(torch.load(cfg['resume_ckpt'], map_location='cpu').get('epoch', -1)) + 1), 1) if Path(cfg['resume_ckpt']).exists() else cfg['epochs']
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=remaining_epochs, eta_min=1e-6)
loss_fn = CompoundLossV2(pos_weight=cfg['pos_weight']).to(device)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda' and cfg['use_amp']))

ema_model = copy.deepcopy(core_model).to(device).eval()
for p in ema_model.parameters():
    p.requires_grad = False

@torch.no_grad()
def update_ema(ema_model, model, decay):
    src = unwrap_model(model)
    ema_params = dict(ema_model.named_parameters())
    src_params = dict(src.named_parameters())
    for name, param in src_params.items():
        ema_params[name].mul_(decay).add_(param.data, alpha=1.0 - decay)
    ema_buffers = dict(ema_model.named_buffers())
    src_buffers = dict(src.named_buffers())
    for name, buf in src_buffers.items():
        ema_buffers[name].copy_(buf)

resume_state = load_hr_resume_state(core_model, ema_model, RUN_DIR)
unfreeze_last_blocks(model, n_last_blocks=cfg['unfreeze_last_blocks'])

if is_main_process():
    print('params M total:', sum(p.numel() for p in core_model.parameters()) / 1e6)
    print('bev upsample factor:', cfg['bev_upsample_factor'])
    print('resume enabled:', resume_state['enabled'])
    print('resume start_epoch:', resume_state['start_epoch'])

    with torch.no_grad():
        batch = next(iter(loader_train))
        images = batch['images'].to(device)
        intr = batch['intrinsics'].to(device)
        c2c = batch['car2cams'].to(device)
        rover_id = batch['rover_id'].to(device)
        dbg = core_model.forward_debug(images, intr, c2c, rover_id)
        print('image_fused:', tuple(dbg['image_fused'].shape), torch.isfinite(dbg['image_fused']).all().item())
        print('bev_raw_hr:', tuple(dbg['bev_raw_hr'].shape), torch.isfinite(dbg['bev_raw_hr']).all().item())
        print('logits_hr:', tuple(dbg['logits_hr'].shape), torch.isfinite(dbg['logits_hr']).all().item())
        print('logits:', tuple(dbg['logits'].shape), torch.isfinite(dbg['logits']).all().item())

cleanup_cuda()
barrier()


In [ ]:
log = list(resume_state['log'])
best_iou = float(resume_state['best_iou'])
best_ema_iou = float(resume_state['best_ema_iou'])
start = time.time() - float(resume_state['elapsed_minutes']) * 60.0
start_epoch = int(resume_state['start_epoch'])

for epoch in range(start_epoch, cfg['epochs']):
    if train_sampler_warm is not None:
        train_sampler_warm.set_epoch(epoch)
    if train_sampler_aug is not None:
        train_sampler_aug.set_epoch(epoch)

    model.train()
    loader = loader_train
    losses = []
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(loader, desc=f'ep{epoch:02d} [BEV2X]', disable=not is_main_process())

    for step, batch in enumerate(pbar):
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt = batch['gt'].to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda' and cfg['use_amp'])):
            logits = model(images, intr, c2c, rover_id)
        logits = logits.float()
        loss, parts = loss_fn(logits, gt)
        loss = loss / cfg['grad_accum']

        if not torch.isfinite(loss):
            raise RuntimeError(f'Non-finite loss detected at epoch={epoch} step={step}: {loss.item()}')

        scaler.scale(loss).backward()
        if (step + 1) % cfg['grad_accum'] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(core_model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            update_ema(ema_model, model, cfg['ema_decay'])

        losses.append(loss.item() * cfg['grad_accum'])
        if is_main_process() and step % 20 == 0:
            pbar.set_postfix(loss=f'{np.mean(losses[-50:]):.3f}', bce=f"{parts['bce']:.2f}")

    if len(loader) % cfg['grad_accum'] != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(core_model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        update_ema(ema_model, model, cfg['ema_decay'])

    scheduler.step()
    barrier()

    if is_main_process():
        cleanup_cuda()
        print('start val model @0.5')
        val_iou_05 = evaluate_iou(core_model, loader_val, threshold=0.5, desc=f'val model ep{epoch:02d}')

        cleanup_cuda()
        print('start val ema @0.5')
        val_iou_05_ema = evaluate_iou(ema_model, loader_val, threshold=0.5, desc=f'val ema ep{epoch:02d}')
        cleanup_cuda()

        elapsed = (time.time() - start) / 60
        row = {
            'epoch': epoch,
            'loss': float(np.mean(losses)) if len(losses) else None,
            'val_iou_05': float(val_iou_05),
            'val_iou_05_ema': float(val_iou_05_ema),
            'minutes': float(elapsed),
        }

        print(f"ep{epoch:02d} | loss {np.mean(losses):.3f} | val@0.5 {val_iou_05:.4f} | ema@0.5 {val_iou_05_ema:.4f} | {elapsed:.1f}m")

        if val_iou_05 > best_iou:
            best_iou = val_iou_05
            torch.save({
                'model_type': 'v62_dinov2_lss_bev2x_cleaned',
                'model': core_model.state_dict(),
                'epoch': epoch,
                'best_iou': float(val_iou_05),
                'best_t': 0.5,
                'rover_vocab': rover_vocab,
                'cfg': cfg,
            }, RUN_DIR / 'best.pt')
            print('  new best model:', val_iou_05)

        if val_iou_05_ema > best_ema_iou:
            best_ema_iou = val_iou_05_ema
            torch.save({
                'model_type': 'v62_dinov2_lss_bev2x_cleaned',
                'model': core_model.state_dict(),
                'ema': ema_model.state_dict(),
                'epoch': epoch,
                'best_ema_iou': float(val_iou_05_ema),
                'best_t_ema': 0.5,
                'rover_vocab': rover_vocab,
                'cfg': cfg,
            }, RUN_DIR / 'ema_best.pt')
            print('  new best ema:', val_iou_05_ema)

        log.append(row)
        pd.DataFrame(log).to_csv(RUN_DIR / 'log.csv', index=False)
        torch.save({
            'model_type': 'v62_dinov2_lss_bev2x_cleaned',
            'model': core_model.state_dict(),
            'ema': ema_model.state_dict(),
            'epoch': epoch,
            'best_iou': float(best_iou),
            'best_ema_iou': float(best_ema_iou),
            'rover_vocab': rover_vocab,
            'cfg': cfg,
        }, RUN_DIR / 'last.pt')

    barrier()


### Notes

- Это **architectural upgrade resume**, а не чистый resume: веса модели и EMA подгружаются из старого `last.pt` только по совпадающим ключам.
- Новые `BEV attention` блоки инициализируются случайно и учатся с отдельным `lr_new_attn`.
- Optimizer/scheduler/scaler специально стартуют заново: их shape/state из старой архитектуры не переносится.
- По умолчанию notebook заточен под kaggle-safe dataset layout и один GPU.
